In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# 1. 定义你的投资组合 (代码和权重)
# GE 在 2024 年 4 月拆分，这里我们使用新的航空航天公司 GE 的代码
# AT&T 的代码是 T
portfolio = {
    'SPY': 0.15,
    'TSLA': 0.10,
    'GOOG': 0.08,
    'JNJ': 0.07,
    'XOM': 0.06,
    'RTX': 0.06,
    'CVX': 0.05,
    'CRM': 0.05,
    'PFE': 0.05,
    'META': 0.05,
    'BOMN': 0.04,
    'INTC': 0.04,
    'CMCSA': 0.04,
    'GE': 0.04,
    'ASLE': 0.02,
    'IBM': 0.03,
    'AAPL': 0.03,
    'PYPL': 0.02,
    'T': 0.02,
}

# 检查权重总和是否为1 (100%)
# assert sum(portfolio.values()) == 1.0, "权重总和不等于1"
print(f"权重总和: {sum(portfolio.values()):.2%}")


# 2. 设置回测参数
start_date = '2019-01-01'
end_date = pd.to_datetime('today').strftime('%Y-%m-%d')
initial_capital = 100000
benchmark_ticker = 'SPY' # 标普500 ETF 作为基准

# 3. 获取股价数据 (使用 'Adj Close' 来自动处理股息和拆分)
tickers = list(portfolio.keys())
all_tickers = tickers + [benchmark_ticker]
data = yf.download(all_tickers, start=start_date, end=end_date)['Adj Close']

# 过滤掉数据不完整的股票 (例如 BOMN 上市较晚)
initial_prices = data.iloc[0].dropna()
valid_tickers = initial_prices.index.tolist()
if benchmark_ticker not in valid_tickers:
    raise ValueError("基准数据在起始日期不可用")

# 重新计算有效组合的权重
valid_portfolio = {k: v for k, v in portfolio.items() if k in valid_tickers}
weight_sum = sum(valid_portfolio.values())
normalized_portfolio = {k: v / weight_sum for k, v in valid_portfolio.items()}

print("\n回测开始时可用的股票及其归一化权重:")
for ticker, weight in normalized_portfolio.items():
    print(f"{ticker}: {weight:.2%}")

# 4. 执行“买入并持有”回测
# 计算初始买入的股数
initial_investment_per_stock = {ticker: initial_capital * weight for ticker, weight in normalized_portfolio.items()}
shares_held = {}
for ticker in normalized_portfolio.keys():
    # 确保 data[ticker].iloc[0] 不是 NaN
    if pd.notna(data[ticker].iloc[0]):
        shares_held[ticker] = initial_investment_per_stock[ticker] / data[ticker].iloc[0]

# 计算每天的投资组合价值
portfolio_values = pd.DataFrame(index=data.index)
for ticker, shares in shares_held.items():
    portfolio_values[ticker] = data[ticker] * shares

portfolio_values['Total'] = portfolio_values.sum(axis=1)

# 5. 计算基准表现
benchmark_initial_price = data[benchmark_ticker].iloc[0]
benchmark_shares = initial_capital / benchmark_initial_price
benchmark_values = data[benchmark_ticker] * benchmark_shares

# 6. 计算和展示结果
# 将所有价值归一化，以便比较
portfolio_normalized = (portfolio_values['Total'] / initial_capital) * 100
benchmark_normalized = (benchmark_values / initial_capital) * 100

# 计算最终回报
final_portfolio_value = portfolio_values['Total'].iloc[-1]
final_benchmark_value = benchmark_values.iloc[-1]
total_return_portfolio = (final_portfolio_value / initial_capital) - 1
total_return_benchmark = (final_benchmark_value / initial_capital) - 1

# 计算年化回报率 (CAGR)
years = (data.index[-1] - data.index[0]).days / 365.25
cagr_portfolio = ( (final_portfolio_value / initial_capital) ** (1/years) ) - 1
cagr_benchmark = ( (final_benchmark_value / initial_capital) ** (1/years) ) - 1

print(f"\n--- 回测结果 ({start_date} to {end_date}) ---")
print(f"初始资本: ${initial_capital:,.2f}")
print("\n--- 你的投资组合 ---")
print(f"最终价值: ${final_portfolio_value:,.2f}")
print(f"总回报率: {total_return_portfolio:.2%}")
print(f"年化回报率 (CAGR): {cagr_portfolio:.2%}")

print("\n--- 标普 500 (SPY) 基准 ---")
print(f"最终价值: ${final_benchmark_value:,.2f}")
print(f"总回报率: {total_return_benchmark:.2%}")
print(f"年化回报率 (CAGR): {cagr_benchmark:.2%}")


# 7. 可视化结果
plt.style.use('seaborn-v0_8-darkgrid')
plt.figure(figsize=(14, 7))
plt.plot(portfolio_normalized.index, portfolio_normalized, label=f'Your Portfolio (CAGR: {cagr_portfolio:.2%})', color='royalblue', linewidth=2)
plt.plot(benchmark_normalized.index, benchmark_normalized, label=f'S&P 500 (SPY) (CAGR: {cagr_benchmark:.2%})', color='grey', linestyle='--')

plt.title('Portfolio Buy-and-Hold Performance vs. S&P 500', fontsize=16)
plt.ylabel('Growth of $100', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend(fontsize=12)
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(100)) # Y轴显示为百分比
plt.grid(True)
plt.show()